# [초급 프로젝트] 4팀_김명환

---
---

# 프로그래밍

In [6]:
!pip install -q gdown
!pip install -q albumentations
!pip install -q ultralytics
!pip install -q -U ultralytics
!pip install -q nbformat
!pip install -q roboflow
!pip install -q opencv-python
!pip install -q opencv-python-headless
!pip install -q wandb
print("로딩완료")

로딩완료


In [7]:
# !wandb login 86a7b8c07184b2efdfb116546a17b1905e41cb5d

In [8]:
# 기본 라이브러리 (중복 제거 및 정리)

# --- Scikit-learn: 데이터 전처리, 모델, 평가 ---
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.datasets import (
    fetch_california_housing, load_iris, make_moons, make_circles,
    load_breast_cancer, load_wine
)
from sklearn import datasets
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, average_precision_score

# --- 이미지 처리 ---
import cv2
from PIL import Image, ImageFilter, ImageDraw
import albumentations as A

# --- PyTorch: 딥러닝 관련 ---
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
# 문제 있는 v2 import 제거하고 필요시에만 개별적으로 import
# from torchvision.transforms import v2, functional as TF
from torchvision.transforms import functional as TF
from torchvision.datasets import CocoDetection
from torch.nn import CrossEntropyLoss
from collections import OrderedDict

# --- COCO 데이터셋 관련 ---
from pycocotools.coco import COCO
from pycocotools import mask as coco_mask

# --- 딥러닝 모델 ---
import timm

# --- 기본 라이브러리 ---
import os
import sys
import re
import csv
import copy
import json
import math
import random
import yaml
import shutil
import requests
import xml.etree.ElementTree as ET
from pathlib import Path

# --- 데이터 분석 및 시각화 ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# --- 시간 관련 ---
from datetime import datetime, timezone, timedelta
import pytz

# --- 진행률 표시 ---
import IPython.display
from tqdm.notebook import tqdm

# --- 시간대 설정 ---
__kst = pytz.timezone('Asia/Seoul')

# --- GPU 설정 ---
__device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
__device_cpu = torch.device('cpu')

# --- 재현 가능한 결과를 위한 시드 설정 ---
np.random.seed(42)
torch.manual_seed(42)
if __device.type == 'cuda':
    torch.cuda.manual_seed_all(42)

print(f"라이브러리 로드 완료 사용장치: {__device}")

라이브러리 로드 완료 사용장치: cuda


In [9]:
from urllib.request import urlretrieve; urlretrieve("https://raw.githubusercontent.com/c0z0c/jupyter_hangul/refs/heads/beta/helper_c0z0c_dev.py", "helper_c0z0c_dev.py")
import importlib
import helper_c0z0c_dev as helper
importlib.reload(helper)

🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.0.2 (v2.x+): 호환성 모드 적용됨
Mounted at /content/drive
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = /content/drive/MyDrive


<module 'helper_c0z0c_dev' from '/content/helper_c0z0c_dev.py'>

In [10]:
from urllib.request import urlretrieve; urlretrieve("https://raw.githubusercontent.com/c0z0c/codeit_ai_health_eat/refs/heads/alpha/src/python_modules/utils/health_ea_utils.py",
                                                    "health_ea_utils.py")
import importlib
import health_ea_utils as heu
importlib.reload(heu)
from health_ea_utils import *

🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.0.2 (v2.x+): 호환성 모드 적용됨
Mounted at /content/drive
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = /content/drive/MyDrive
🌐 https://c0z0c.github.io/jupyter_hangul
ℹ️ NumPy 2.0.2 (v2.x+): 호환성 모드 적용됨
Mounted at /content/drive
✅ 설정 완료: 한글 폰트, plt 전역 등록, pandas 확장, 캐시 기능
pd commit 저장 경로 = /content/drive/MyDrive


# 데이타 다운로드

In [11]:
# download_files={
#     'yolo_label_one_class' : r'https://drive.google.com/file/d/177_86k4BuT6JnFnq7ZHJtEjp7jaRbCl2/view?usp=sharing',
#     'yolo_label' : r'https://drive.google.com/file/d/1nc-WFcw7lCS7s7VGzN9Kxh80PiBBggez/view?usp=sharing',
#     'yolo_resize_one_class' : r'https://drive.google.com/file/d/1Ak0EvkMnuwvcAFvTO-zovIgVcNlROjsS/view?usp=sharing',
#     'yolo_resize' : r'https://drive.google.com/file/d/1kpo57qOJhEhrkuzUCEh57ILB5xSPVoFv/view?usp=sharing',
# }

# download_files={
#     'yolo_label' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODU5OTkzNTMyOHxGfDA&svcType=MYBOX-WEB&time=1757776010785',
#     'yolo_label_one_class' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODYzOTg5NDExMnxGfDA&svcType=MYBOX-WEB&time=1757776673721',
#     'yolo_resize_one_class' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODgwNjk2NDMyMHxGfDA&svcType=MYBOX-WEB&time=1757780142635',
#     'yolo_resize' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODY4MDc2MjQ2NHxGfDA&svcType=MYBOX-WEB&time=1757780177672',
# }

download_files={
    # 'yolo_label' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODU5OTkzNTMyOHxGfDA&svcType=MYBOX-WEB&time=1757776010785',
    # 'yolo_label_one_class' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODYzOTg5NDExMnxGfDA&svcType=MYBOX-WEB&time=1757776673721',
    # 'yolo_resize_one_class' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODgwNjk2NDMyMHxGfDA&svcType=MYBOX-WEB&time=1757780142635',
    # 'yolo_resize' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc1ODY4MDc2MjQ2NHxGfDA&svcType=MYBOX-WEB&time=1757780177672',
    'yolo_noresize' : r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc2NDA0ODY2ODI1NnxGfDA&svcType=MYBOX-WEB&time=1757851996107',
    #'yolo_noresize_one_class':r'https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc2NDgwMDcxODk0NHxGfDA&svcType=MYBOX-WEB&time=1757893856220',
}
# yolo_noresize = https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc2NDA0ODY2ODI1NnxGfDA&svcType=MYBOX-WEB&time=1757851996107
# yolo_noresize_one_class = https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc2NDgwMDcxODk0NHxGfDA&svcType=MYBOX-WEB&time=1757893856220



In [12]:
import gdown
def download_gdrive_file(url, output_path, ignore=True):
    # 공유 링크에서 파일 ID 추출
    if os.path.exists(output_path):
        if ignore:
            os.remove(output_path)
        else:
            return

    file_id_match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if not file_id_match:
        raise ValueError("Google Drive 파일 ID를 찾을 수 없습니다.")
    file_id = file_id_match.group(1)
    gdown.download(f"https://drive.google.com/uc?id={file_id}", output_path, quiet=False)

def download_http(url, target, ignore=True):
    """
    HTTP 파일 다운로드 함수 (진행률 표시)
    url: 다운로드할 파일 URL
    target: 저장할 파일 경로
    ignore: True면 기존 파일 삭제 후 다운로드, False면 파일 있으면 건너뜀
    """
    if os.path.exists(target):
        if ignore:
            os.remove(target)
        else:
            print(f"이미 파일이 존재합니다: {target}")
            return target

    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(target, 'wb') as file, tqdm(
        desc=f"Downloading {os.path.basename(target)}",
        total=total,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
        ascii=True
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)
    print(f"다운로드 완료: {target}")
    return target

# local_code_it_ai04 = os.path.join( '~/.cache/' if helper.is_colab else Path.cwd().drive + '\\'
#                                   ,'temp'
#                                   , 'code_it_ai04')

if helper.is_colab:
    local_code_it_ai04 = os.path.join( '/content/', 'code_it_ai04')
else:
    local_code_it_ai04 = os.path.join( Path.cwd().drive + '\\', 'temp', 'code_it_ai04')

print("local_code_it_ai04:", local_code_it_ai04)

os.makedirs(local_code_it_ai04, exist_ok=True)  # 폴더 생성 코드 추가
unzip_paths = []
for key, url in download_files.items():
    print(f"{key}: {url}")
    zipfile = os.path.join(local_code_it_ai04, f'{key}.zip')
    unzip_path = os.path.join(local_code_it_ai04, f'{key}.zip.unzip')
    if os.path.exists(unzip_path):
        print(f"이미 압축해제된 폴더가 존재합니다: {unzip_path}")
        print('unzipfile:', unzip_path)
        unzip_paths.append(unzip_path)
        continue
    #download_gdrive_file(url, os.path.join(local_code_it_ai04, f'{key}.zip'), ignore=False)
    download_http(url, zipfile, ignore=False)
    unzip_path_list = heu.unzip([os.path.join(local_code_it_ai04, f'{key}.zip')])
    # for p in unzip_path_list:
    #     unzip_paths.append(p)
    print('unzip_path_list:', unzip_path_list)
    unzip_paths.extend(unzip_path_list)


local_code_it_ai04: /content/code_it_ai04
yolo_noresize: https://fs.mybox.naver.com/file/download.api?resourceKey=YzB6MGN8MzQ3MjU5Nzc2NDA0ODY2ODI1NnxGfDA&svcType=MYBOX-WEB&time=1757851996107


다운로드 완료: /content/code_it_ai04/yolo_noresize.zip


압축 해제 중: yolo_noresize.zip: 100%|██████████| 3822/3822 [00:02<00:00, 1748.30file/s]

압축 해제 완료: /content/code_it_ai04/yolo_noresize.zip.unzip
unzip_path_list: ['/content/code_it_ai04/yolo_noresize.zip.unzip']


### > 설정 < 플레그

In [13]:
yolo_dataset_path =unzip_paths[0]
yaml_path = os.path.join(yolo_dataset_path, "dataset.yaml")

print(unzip_paths)
print(yolo_dataset_path)
print(yaml_path)

with open(yaml_path, 'r') as f:
    dataset_config = yaml.safe_load(f)
test_images = [str(p) for p in Path(os.path.join(yolo_dataset_path, dataset_config['test'])).glob("*.png")]
test_images = [str(p) for p in Path(os.path.join(yolo_dataset_path, dataset_config['test'])).glob("*.jpg")]

['/content/code_it_ai04/yolo_noresize.zip.unzip']
/content/code_it_ai04/yolo_noresize.zip.unzip
/content/code_it_ai04/yolo_noresize.zip.unzip/dataset.yaml


In [14]:
import os, sys
from pathlib import Path
user_dir = Path(drive_root()) / "codeit_ai_health_eat" / "scripts" / "김명환"
sys.path.insert(0, str(user_dir.resolve()))
import PillAnalysisEngine as PillAnalysisEngine
import importlib; importlib.reload(PillAnalysisEngine)
from PillAnalysisEngine import PillAnalysisEngine

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [15]:
# pillEngine_test = PillAnalysisEngine()


In [16]:
#df_drug = pillEngine_test.database['df_drug']
#df_drug

In [17]:
def save_all_detection_images():
    """모든 테스트 이미지에 대해 검출 결과를 이미지로 저장하고 CSV 파일도 생성"""
    import pandas as pd
    import matplotlib.patches as mpatches

    # 결과 저장 폴더 생성
    timestamp = datetime.now(__kst).strftime("%Y%m%d_%H%M%S")
    result_folder_name = f'test_result_{timestamp}'

    output_folder = os.path.join(drive_root(),
                                'codeit_ai_health_eat',
                                'scripts',
                                '김명환',
                                result_folder_name)

    os.makedirs(output_folder, exist_ok=True)

    # print(f"검출 결과 이미지 저장 및 CSV 생성 시작...")
    # print(f"저장 폴더: {output_folder}")

    # CSV 관련 변수
    rows_list = []
    csv_rows = []
    csv_headers = ['annotation_id', 'image_id', 'category_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h', 'score']

    saved_images = []
    annotation_id = 1

    test_images_simple = test_images
    #test_images_simple = test_images[:1]
    #test_images_sample = [r'D:\dataset\kaggle_code_it_data\ai04-level1-project.zip.unzip\test_images\411.png',]

    pillEngine = PillAnalysisEngine()
    for i, image_path in enumerate(tqdm(test_images_simple, desc="이미지 검출 및 저장")):

        result_json = pillEngine.analyze_image(image_path)
        basename = os.path.basename(image_path).split('.')[0]
        pillEngine.save_img(pillEngine.image_result(result_json), os.path.join(output_folder, f"result_{basename}.jpg"))

        annotation_id, rows = pillEngine.result_json_to_csv_rows(result_json, image_path, annotation_id)
        rows_list.extend(rows)
        #print(annotation_id, rows)

    pd.DataFrame(rows_list, columns=csv_headers).head_att(10)

    # CSV 파일로 저장
    csv_path = os.path.join(output_folder, f"result_2way_{timestamp}.csv")
    os.makedirs(output_folder, exist_ok=True)
    pd.DataFrame(rows_list, columns=csv_headers).to_csv(csv_path, index=False)
    print(f"CSV 파일 저장 완료: {csv_path}")

# 실행
#output_path, saved_images, results_df, csv_path = save_all_detection_images()

save_all_detection_images()

/content/drive/MyDrive/codeit_ai_health_eat
/content/drive/MyDrive/codeit_ai_health_eat/src/python_modules/modeling
/content/drive/MyDrive/codeit_ai_health_eat/src/python_modules/data
categorys: [249, 572, 1865, 1899, 2482, 3350, 3482, 3543, 3613, 3742, 3831, 4377, 4542, 4999, 5001, 5093, 5390, 5885, 6191, 6562, 6696, 6834, 10220, 10223, 10486, 11219, 12080, 12246, 12419, 12637, 12777, 13003, 13160, 13394, 13899, 15279, 15709, 16205, 16231, 16234, 16261, 16547, 16550, 16687, 18109, 18146, 18356, 19231, 19551, 19606, 19860, 19880, 20003, 20013, 20237, 20258, 20752, 20851, 20876, 21025, 21324, 21770, 22073, 22346, 22361, 22626, 23202, 23222, 23318, 24751, 24849, 24940, 25199, 25366, 25437, 25468, 26454, 26992, 27652, 27732, 27776, 27925, 27992, 28423, 28762, 29344, 29450, 29666, 29710, 29870, 30307, 30849, 31704, 31862, 31884, 32309, 33008, 33025, 33207, 33877, 33879, 34596, 35205, 35658, 36636, 37776, 38161, 38722, 38926, 38953, 38971, 41148, 41767, 43232, 44198, 44265, 44833, 53383]
Tr

이미지 검출 및 저장: 100%|██████████| 843/843 [10:31<00:00,  1.34it/s]

   annotation_id image_id category_id bbox_x bbox_y bbox_w bbox_h  score
 0             1      571       16550    538     51    306    457      1
 1             2      571       21025     92    833    180    175 0.9985
 2             3      571       24849    125    137    173    276 0.9986
 3             4      571        1899    582    867    212    143 0.9997
 4             5     1367        3482    561    810    284    172 0.9996
 5             6     1367       27652     61    811    217    201      1
 6             7     1367       29666    604    176    215    213      1
 7             8     1367       22346    108    226    199    191 0.9927
 8             9       65       20013      3    634    334    340      1
 9            10       65        3350    355    155    174    179 0.9924
CSV 파일 저장 완료: /content/drive/MyDrive/codeit_ai_health_eat/scripts/김명환/test_result_20250922_155619/result_2way_20250922_155619.csv
